# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Storage Solutions (Neo4j)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
!pip install graphframes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 KB 438.1 kB/s eta 0:00:0000:0100:01


In [ ]:
from spark_utils import SparkUtils
neo4j_connector = "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"
su = SparkUtils(spark_packages=neo4j_connector)
su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 01:36:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


AttributeError: 'SparkSession' object has no attribute 'config'

26/03/19 02:19:12 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:279)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:741)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:141)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:104)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:101)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$spark$rpc$netty$MessageLoop$$receiveLoop(MessageLoop.scala:76)
	at org.apache.spark.rpc.netty.MessageLoop$$anon$1.run(MessageLoop.scala:42)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/jav

# Create GraphFrames

In [ ]:
from graphframes import GraphFrame

vertices = su.spark.createDataFrame([
  ("a", "Alice"),
  ("b", "Bob"),
  ("c", "Carol")
], ["id", "name"])

edges = su.spark.createDataFrame([
  ("a","b","follows"),
  ("b","c","follows"),
  ("c","a","follows")
], ["src","dst","relationship"])

g = GraphFrame(vertices, edges)
g.vertices.show()
g.edges.show()

/opt/spark/python/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


+---+-----+
| id| name|
+---+-----+
|  a|Alice|
|  b|  Bob|
|  c|Carol|
+---+-----+

+---+---+------------+
|src|dst|relationship|
+---+---+------------+
|  a|  b|     follows|
|  b|  c|     follows|
|  c|  a|     follows|
+---+---+------------+



# Core Graph Algorithms
## PageRank

In [4]:
results = g.pageRank(resetProbability=0.15, maxIter=10)

# Ranked vertices (descending by importance)
results.vertices \
  .select("id", "name", "pagerank") \
  .orderBy("pagerank", ascending=False) \
  .show()

/opt/spark/python/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+-----+--------+
| id| name|pagerank|
+---+-----+--------+
|  a|Alice|     1.0|
|  b|  Bob|     1.0|
|  c|Carol|     1.0|
+---+-----+--------+



# Label Propagation

In [5]:
lpa = g.labelPropagation(maxIter=5)
lpa.show()

+---+-----+-----+
| id| name|label|
+---+-----+-----+
|  a|Alice|    0|
|  b|  Bob|    0|
|  c|Carol|    0|
+---+-----+-----+



# Triangle counting

In [8]:
triangle_count = g.triangleCount()
triangle_count.show()

/opt/spark/python/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+-----+---+-----+
|count| id| name|
+-----+---+-----+
|    1|  a|Alice|
|    1|  b|  Bob|
|    1|  c|Carol|
+-----+---+-----+



# Degrees Distribution

## InDregree

In [10]:
in_deg = g.inDegrees.join(vertices, "id")
in_deg.show()

/opt/spark/python/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+--------+-----+
| id|inDegree| name|
+---+--------+-----+
|  b|       1|  Bob|
|  a|       1|Alice|
|  c|       1|Carol|
+---+--------+-----+



## OutDegree

In [11]:
out_deg = g.outDegrees.join(vertices, "id")
out_deg.show()

/opt/spark/python/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+---------+-----+
| id|outDegree| name|
+---+---------+-----+
|  a|        1|Alice|
|  b|        1|  Bob|
|  c|        1|Carol|
+---+---------+-----+



# Write data to a Neo4j Graph

## Neo4j setup
### Install Neo4j with Docker

Go to **spark** directory and run:

```
docker run \
    -d --restart always \
    --publish=7474:7474 --publish=7687:7687 \
    --env NEO4J_AUTH=neo4j/neo4j@1234 \
    --volume=./data_neo4j:/data \
    --name neo4j-iteso \
    --network spark-cluster_default \
    neo4j:community-trixie
```

## Write GraphFrames into Neo4J

In [12]:
neo4j_url = "bolt://neo4j-iteso:7687"
neo4j_user = "neo4j"
neo4j_passwd = "neo4j@1234"

g.vertices.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":User") \
  .option("node.keys", "id") \
  .save()

print(f"{g.vertices.count()} verticess wrote in Neo4j")


g.edges.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("relationship", "FOLLOWS") \
  .option("relationship.save.strategy", "keys") \
  .option("relationship.source.labels", ":User") \
  .option("relationship.source.save.mode", "match") \
  .option("relationship.source.node.keys", "src:id") \
  .option("relationship.target.labels", ":User") \
  .option("relationship.target.save.mode", "match") \
  .option("relationship.target.node.keys", "dst:id") \
  .save()

print(f"{g.edges.count()} edges wrote in Neo4j")

3 verticess wrote in Neo4j
3 edges wrote in Neo4j


In [13]:
su.spark.stop()